In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from gdynia_srodmiescie import loading as ld
from gdynia_srodmiescie import building_features as bf
from gdynia_srodmiescie import street_metrics as sm
from gdynia_srodmiescie import styling as st
from gdynia_srodmiescie import cleaning as cl

In [ ]:
short = ld.load_df_short("../../data/raw/srodmiescie_raw_object_id.csv", dayfirst=False)
category_map = ld.load_df_long(
    "../../data/reference/category_map.csv", dtype=ld.CATEGORY_COLUMN_TYPES
)

In [ ]:
short_fin = (
    short.pipe(cl.drop_missing, column="ground_floor_function")
    .pipe(
        cl.replace_to_space, column="ground_floor_function", to_replace="_", lower=True
    )
    .pipe(
        cl.replace_to_space, column="upper_floor_function", to_replace="_", lower=True
    )
    .pipe(cl.fix_adresses)
    .pipe(cl.fix_styles)
    .pipe(cl.merge_by_same_obj_key)
)

long_fin = short_fin.pipe(cl.long_df_from_short).pipe(cl.add_categories, category_map)

In [ ]:
t = pd.crosstab(short_fin["arch_style"], short_fin["condition"], normalize="index")
st.plot_heatmap(
    t,
    "styl_stan.png",
    "Building condition divided by styles",
    x_label="building condition(1-poor, 3-great)",
    y_label="style",
)

In [ ]:
st.plot_scatter()

In [ ]:
short_fin.to_csv(
    "../../data/processed/fixed/srodmiescie_short.csv", sep=";", encoding="utf-8"
)
long_fin.to_csv(
    "../../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    encoding="utf-8",
)